# 🌊 AI Urban Flood & Traffic Congestion Monitor (Official SYNEXAN)
### 100% Giao diện Gradio gốc ở Local (Tesla T4 GPU + Cloudflare Tunnel Siêu Tốc)
Triển khai tối ưu trên Kaggle GPU (Tesla T4) + Cloudflare Tunnel (Băng thông KHÔNG GIỚI HẠN).
Tự động kết nối và đồng bộ siêu tốc về Web App trên Vercel: https://frontend-flood.vercel.app


In [ ]:
# 1. Cài đặt các thư viện cần thiết tinh gọn & Tải Cloudflare Tunnel
import os, subprocess, urllib.request

# Kaggle đã có sẵn torch, cuda, opencv, pillow, pandas, nest-asyncio
# Cài đặt chuẩn hóa tương thích tuyệt đối cho Gradio 4.44.0 + Ultralytics + FastAPI
!pip install -q --no-cache-dir --prefer-binary "gradio==4.44.0" "pydantic<2.10.0" "huggingface_hub<0.26.0" ultralytics lapx fastapi uvicorn python-multipart

if not os.path.exists("/tmp/cloudflared"):
    opener = urllib.request.build_opener()
    opener.addheaders = [('User-Agent', 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)')]
    urllib.request.install_opener(opener)
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "/tmp/cloudflared")
    os.chmod("/tmp/cloudflared", 0o755)


In [ ]:
# 2. Đồng bộ mã nguồn Backend siêu tốc từ GitHub SYNEXAN/FrontendFlood (--depth 1)
import os, sys

!rm -rf /kaggle/working/FrontendFlood
!git clone --depth 1 -q https://github.com/SYNEXAN/FrontendFlood.git /kaggle/working/FrontendFlood

os.chdir("/kaggle/working/FrontendFlood/backend")
sys.path.append("/kaggle/working/FrontendFlood/backend")
print("✅ Thư mục backend hiện tại:", os.getcwd())


In [ ]:
# 3. Tải song song Trọng số mô hình Deep Learning (ThreadPoolExecutor)
import os, urllib.request, time
from concurrent.futures import ThreadPoolExecutor

os.makedirs("models", exist_ok=True)
models_info = [
    ("models/vietnam_flood_yolov8s_seg.pt", "https://huggingface.co/spaces/thietnt/AI-Urban-Flood-Traffic-Monitor/resolve/main/models/vietnam_flood_yolov8s_seg.pt", 20000000),
    ("models/yolov8s_vehicle.pt", "https://huggingface.co/spaces/thietnt/AI-Urban-Flood-Traffic-Monitor/resolve/main/models/yolov8s_vehicle.pt", 20000000)
]

def download_one(path, url, min_size):
    if os.path.exists(path) and os.path.getsize(path) >= min_size:
        print(f"✅ Mô hình {path} đã có sẵn ({os.path.getsize(path)} bytes).")
        return True
    for attempt in range(1, 4):
        try:
            print(f"⬇️ Đang tải {path} (lần {attempt}/3)...")
            if os.path.exists(path):
                os.remove(path)
            urllib.request.urlretrieve(url, path)
            if os.path.getsize(path) >= min_size:
                print(f"✅ Tải thành công {path} ({os.path.getsize(path)} bytes).")
                return True
        except Exception as e:
            print(f"❌ Lỗi tải {path}: {e}")
            time.sleep(1)
    return False

with ThreadPoolExecutor(max_workers=2) as executor:
    futures = [executor.submit(download_one, path, url, min_size) for path, url, min_size in models_info]
    for f in futures:
        f.result()

import shutil
custom_vn_weight = "/kaggle/working/FrontendFlood/models/yolov8s_vn_vehicle_best.pt"
if os.path.exists(custom_vn_weight) and os.path.getsize(custom_vn_weight) > 15000000:
    if os.path.exists("models/yolov8s_vehicle.pt") and not os.path.exists("models/yolov8s_vehicle_backup.pt"):
        shutil.copy2("models/yolov8s_vehicle.pt", "models/yolov8s_vehicle_backup.pt")
        print("📦 Đã sao lưu trọng số cũ tại models/yolov8s_vehicle_backup.pt")
    shutil.copy2(custom_vn_weight, "models/yolov8s_vehicle.pt")
    print("🚗 Đã kích hoạt trọng số phương tiện mới (4 lớp VN - mAP50=92.79%) tại models/yolov8s_vehicle.pt")

print("✅ Sẵn sàng các file mô hình:")
!ls -lh models/


In [ ]:
# 4. Khởi động Auto-Heal Supervisor (Cloudflare Tunnel + Gradio GPU + 15s Heartbeat)
import os, sys, subprocess

os.environ['IDLE_TIMEOUT_SECONDS'] = '43200'
os.environ['PYTHONUNBUFFERED'] = '1'

launcher_script = '/kaggle/working/FrontendFlood/scripts/kaggle_backend_launcher.py'
if not os.path.exists(launcher_script):
    subprocess.run(['git', '-C', '/kaggle/working/FrontendFlood', 'pull', '-q'], check=False)

print('🚀 Đang khởi chạy Bộ điều phối Auto-Heal Supervisor mới nhất từ GitHub...')
subprocess.run([sys.executable, launcher_script], check=False)
